# 2c - Minimum risk, with AI

**Time:** 45 minutes build + 15 minutes check + 10 minutes email  |  **AI: writes code from YOUR spec** - you explain every line

**What you will do**
- Write a spec (a short written brief of what you want): Goal, Inputs, Units and conventions, Output, Check.
- Ask Gemini to write the code from your spec. Paste it here.
- Explain every line in your own words.
- Then check the answer, with plain loops and with your own eyes.

The code is written by AI. The decision is yours: what you ask for decides what you get.

## The problem

The client agrees to move his money into three funds, **A, B and C** (the same as in 2a and 2b: we can check every number by hand). He asks: **which mix of A, B and C has the lowest possible risk?** Run the code cell below first: it loads the data and the risk function.

You will ask AI for the code. What you ask decides what you get. Here is the same request written twice, for the risk function you used in 2b:

| | Weak spec | Good spec |
|---|---|---|
| **Goal** | "code for portfolio risk" | Volatility of a portfolio of funds |
| **Inputs** | (not said) | weights, vols, correlation matrix (arrays) |
| **Units and conventions** | (not said) | decimals (0.15 = 15%), annual, weights sum to 1 |
| **Output** | (not said) | one number, the annual volatility |
| **Check** | (not said) | all correlations = 1 gives the weighted average of the vols |

With the weak spec, AI guesses every blank: percent or decimals, daily or annual, variance or volatility. Each guess can be wrong without any error message.

In [ ]:
import numpy as np

# Given: the same data as in 2b
vol = np.array([0.15, 0.20, 0.25])    # volatilities of A, B, C
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])


# Given: your function from 2b (you will need it in the check)
def portfolio_vol(weights, vol, corr):
    if round(np.sum(weights), 6) != 1:
        raise ValueError("Weights must sum to 1.")
    cov = np.diag(vol) @ corr @ np.diag(vol)
    return np.sqrt(weights @ cov @ weights)

## Step 1 - Write your spec

Now write a **good spec** for the client's question: the mix of A, B and C with the lowest risk. Double-click this cell and replace each `...` with your text. Press Shift+Enter to close it. Be precise: AI fills every gap with its own guess.

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Units and conventions** | ... |
| **Output** | ... |
| **Check** | ... |

## Step 2 - Ask Gemini, paste the code

1. Open Gemini (gemini.google.com, or the Gemini button in Colab).
2. Paste your spec, then ask: *"Write Python code with NumPy for this spec."*
3. Paste the code in the cell below.
4. Make the code leave the weights in a variable called `ai_weights` (an array of 3 numbers), and print the weights and the volatility. If Gemini gave other names, rename them.

**Stuck?** If the code fails, or there is no `ai_weights` after you run it, do not skip the checks. Ask your neighbour, or run the backup cell below.

Run the cell.

In [ ]:
# PASTE the code Gemini wrote here.
# The inputs vol and corr are defined above. Keep the variable name ai_weights.

### Backup: a classmate's AI answer

Run the cell below **only if** your own cell above did not work, or did not create `ai_weights`. It is the code that most AI tools write for this problem. You still explain every line in step 3.

If your own code works, skip this cell for now. You come back to it at the end.

In [ ]:
# BACKUP ONLY: a classmate's AI answer. Run it only if your own code did not work.
cov = np.diag(vol) @ corr @ np.diag(vol)      # covariance matrix, as in 2a
inv_cov = np.linalg.inv(cov)                  # the inverse of the covariance matrix (new, not part of the course)
ones = np.ones(3)                             # one 1 per fund
ai_weights = inv_cov @ ones / (ones @ inv_cov @ ones)   # the textbook minimum-variance formula
ai_vol = np.sqrt(ai_weights @ cov @ ai_weights)

print("Weights:", ai_weights)
print(f"Volatility: {ai_vol:.2%}")

## Step 3 - Explain every line

Rule of the course: you must be able to explain every line you submit. For each line of the code above, write what it does, in your own words and in finance words.

Double-click here and write: ...

## Check - Is the answer right?

AI wrote code, and it ran. That is not a check. Three questions, then a brute force.

(Checks card, entries 7 and 8: see the link at the bottom of this notebook.)

### Question 1 - Do the weights sum to 1?

Add up the weights in `ai_weights` with `np.sum`.

In [ ]:
total = ...   # YOUR CODE
print(f"Sum of the weights: {total:.4f}")

### Question 2 - Is the volatility below the lowest single fund?

The lowest single fund is A, at 15%. A portfolio of the three funds should do at least as well.

Call your function `portfolio_vol` from 2b on `ai_weights`, with `vol` and `corr`.

In [ ]:
check_vol = ...   # YOUR CODE
print(f"Volatility: {check_vol:.2%}")
print("Below 15%?", check_vol < 0.15)

### Question 3 - Look at each weight

Do not only check the total. Look at every weight, one by one. A weight is a position in a fund: a positive weight is a purchase, a negative weight is a short position (a bet that the fund will fall).

In [ ]:
print(f"Fund A: {ai_weights[0]:.1%}")
print(f"Fund B: {ai_weights[1]:.1%}")
print(f"Fund C: {ai_weights[2]:.1%}")

**Your answer:** is any weight negative? If yes, what does a negative weight mean in real life? Can this client do that? Did your spec say so? Write 3 to 4 lines here: ...

### Brute force: try every long-only portfolio

A second method, **brute force**: try every possibility and keep the best. No clever formula, only plain loops from Session 1. The code below tries every portfolio in steps of 1% with no negative weights (this is called **long-only**): 5,151 portfolios, each one through your `portfolio_vol`. It keeps the one with the lowest volatility.

The code is given. Read the comments, then make a prediction before you run it.

**Your prediction:** will brute force find the same weights as the AI? A lower volatility, or a higher one?

In [ ]:
best_vol = 1       # start higher than any real vol (100%)
best_w = None      # None means: nothing yet
count = 0

for a in range(0, 101):                  # percent in A: 0 to 100
    for b in range(0, 101 - a):          # percent in B: what is left after A
        c_pct = 100 - a - b              # the rest goes in C
        trial = np.array([a, b, c_pct]) / 100    # percent -> decimals
        v = portfolio_vol(trial, vol, corr)
        count = count + 1
        if v < best_vol:                 # better than the best so far? keep it
            best_vol = v
            best_w = trial

print(f"Portfolios tried: {count:,}")
print(f"Best weights: A {best_w[0]:.0%}, B {best_w[1]:.0%}, C {best_w[2]:.0%}")
print(f"Best volatility: {best_vol:.2%}")

You should get **65 / 35 / 0**, volatility **12.56%**, 5,151 portfolios.

**Explain it:** in 2 lines, what do the two `for` lines do, and what does the `if` do? Double-click here and write: ...

### Your conclusion

The AI may say one thing, the brute force another. **Why do the two answers differ?** What should your spec have said? Write 3 to 4 lines here: ...

Note: if your AI's answer has no negative weight, do not stop here. Compare it with the brute force, and check what method and what constraint it used. Then run the backup cell (a classmate's answer) and run Question 3 again: same problem, a different spec, a different answer. Which one would you send to the MD?

## Answer the MD

Back to the client's three questions. The cell below gives you the numbers for his 6 tech stocks and for 20 stocks (same data and method as the opener of 2a, and the simulation of cell 9 with 9% expected return). Run it.

Then write a **4-line email** to your MD, in finance words, with numbers: how risky the client is today, how much diversifying would cut, where he could be in a year, and what you recommend. No code in the email.

In [ ]:
import pandas as pd

prices = pd.read_csv("https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/stocks_20.csv", index_col="Date")
daily_20 = prices.pct_change().dropna().to_numpy()
stock_vol = daily_20.std(axis=0) * 252 ** 0.5
stock_corr = np.corrcoef(daily_20, rowvar=False)
cov20 = np.diag(stock_vol) @ stock_corr @ np.diag(stock_vol)

for label, wts in [("6 tech stocks", np.array([1/6] * 6 + [0] * 14)), ("20 stocks", np.ones(20) / 20)]:
    v = np.sqrt(wts @ cov20 @ wts)
    rng = np.random.default_rng(42)
    final = 100 * np.prod(1 + rng.normal(0.09 / 252, v / 252 ** 0.5, size=(252, 5000)), axis=0)
    print(f"{label}: vol {v:.1%}, bad year (1 in 20) {np.percentile(final, 5):.0f} per 100, chance of a loss {(final < 100).mean():.0%}")

**Your email to the MD.** Double-click here and write 4 lines: ...

## Go further (if you finish early)

Load the 20 real stocks from the top of notebook 2a (the code is in the cell below). AI may help, but you must explain every line.

In [ ]:
import pandas as pd
import math

prices = pd.read_csv("https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/stocks_20.csv", index_col="Date")
daily_20 = prices.pct_change().dropna().to_numpy()
stock_vol = daily_20.std(axis=0) * 252 ** 0.5
stock_corr = np.corrcoef(daily_20, rowvar=False)
print(prices.shape)

### Go further 1 - Minimum risk for 20 stocks

Run the backup formula (from step 2) on the 20 stocks. How many weights are negative? Which stocks get the biggest weights, and why those?

In [ ]:
# YOUR CODE

### Go further 2 - Can brute force do 20 stocks?

For 3 funds in steps of 1%, brute force tried 5,151 portfolios. For 20 stocks, the count is `math.comb(119, 19)`. At one million portfolios per second, how many **years** would it take?

In [ ]:
# YOUR CODE

### Go further 3 - Ask for long-only

Write a spec for the minimum-risk portfolio of the 20 stocks **with no short positions**, and ask Gemini for code. It will likely use `scipy.optimize`. Check its answer: weights sum to 1? No negative weight? Volatility between 11.8% (with shorts) and 15.7% (equal weights)?

In [ ]:
# PASTE Gemini's code here, then check it

## What you learned

**Python**
- **Function**: `portfolio_vol(weights, vol, corr)` reused inside a loop, with its **Guard** (`raise ValueError`) still checking every portfolio.
- **Array** and **Matrix product** in AI code: `inv_cov @ ones`, `ones @ inv_cov @ ones`. **Diagonal matrix** (`np.diag`), **Square root** (`np.sqrt`), **2D array + index** and **Shape** all appear in the code you were asked to read.
- **For loop** inside a for loop, with `range` and `if v < best_vol`: you read and explained a brute force over 5,151 portfolios.
- **Import**: `import numpy as np`. Indexing `ai_weights[2]` reads one weight.

**Finance**
- A spec is five lines: Goal, Inputs, Units and conventions, Output, Check. A weak spec leaves blanks; AI fills every blank with its own guess.
- The answer to the client is an email with numbers, not code: 29% today, 16% diversified, a bad year at 65 vs 84, a recommendation.
- Weights that sum to 1 and a low volatility are not enough. A negative weight is a short position: check the constraints, not only the number.
- The clever method (72 / 50 / -22, vol 12.0%) and the brute force (65 / 35 / 0, vol 12.56%) disagree. The spec decides the answer.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right